### DATA ANALYSIS

In [ ]:
USE OLISTDB;

SELECT TOP 1  * FROM olist_orders_dataset;
SELECT TOP 1 * FROM olist_order_items_dataset;
SELECT TOP 1 * FROM olist_customers_dataset;
SELECT TOP 1 * FROM olist_products_dataset;
SELECT TOP 1 * FROM olist_sellers_dataset;
SELECT TOP 1 * FROM olist_order_payments_dataset;
SELECT TOP 1 * FROM olist_order_reviews_dataset;
SELECT TOP 1 * FROM olist_geolocation_dataset;
SELECT TOP 1 * FROM olist_category_name_dataset;


Msg 208, Level 16, State 1, Line 11
Invalid object name 'olist_category_name_dataset'.

(1 row affected)
(1 row affected)
(1 row affected)
(1 row affected)
(1 row affected)
(1 row affected)
(1 row affected)
(1 row affected)

order_id                         | customer_id                      | order_status | order_purchase_timestamp    | order_approved_at           | order_delivered_carrier_date | order_delivered_customer_date | order_estimated_delivery_date
---------------------------------+----------------------------------+--------------+-----------------------------+-----------------------------+------------------------------+-------------------------------+------------------------------
e481f51cbdc54678b7cc49136f2d6af7 | 9ef432eb6251297304e76186b10a928d | delivered    | 2017-10-02 10:56:33.0000000 | 2017-10-02 11:07:15.0000000 | 2017-10-04 19:55:00.0000000  | 2017-10-10 21:25:13.0000000   | 2017-10-18 00:00:00.0000000  
(1 row)

order_id                         | order_item_id | p

### WHAT ARE THE TOP 5 PRODUCT CATEGORIES BY TOTAL REVENUE?

In [ ]:
WITH sales_revenue AS(
SELECT TOP 5
COALESCE(pct.product_category_name_english, op.product_category_name) AS product_category,
ROUND(SUM(oi.price), 2) AS revenue
FROM olist_order_items_dataset oi
JOIN olist_products_dataset op
ON oi.product_id = op.product_id
LEFT JOIN product_category_name_translation pct
ON op.product_category_name=pct.product_category_name
GROUP BY COALESCE(pct.product_category_name_english, op.product_category_name)
ORDER BY revenue DESC
)
SELECT * FROM sales_revenue;


(5 rows affected)

product_category      | revenue   
----------------------+-----------
health_beauty         | 1258681.34
watches_gifts         | 1205005.68
bed_bath_table        | 1036988.68
sports_leisure        | 988048.97 
computers_accessories | 911954.32 
(5 rows)

Total execution time: 00:00:00.403

### WHICH CATEGORIES SELL THE MOST ITEMS?

In [ ]:
SELECT TOP 5
COALESCE(pct.product_category_name_english, p.product_category_name) AS product_category,
COUNT(*) AS items_sold
FROM olist_order_items_dataset oi
JOIN olist_products_dataset p
ON oi.product_id = p.product_id
LEFT JOIN product_category_name_translation pct
ON p.product_category_name=pct.product_category_name
GROUP BY COALESCE(pct.product_category_name_english, p.product_category_name)       
ORDER BY items_sold DESC;

(5 rows affected)

product_category      | items_sold
----------------------+-----------
bed_bath_table        | 11115     
health_beauty         | 9670      
sports_leisure        | 8641      
furniture_decor       | 8334      
computers_accessories | 7827      
(5 rows)

Total execution time: 00:00:00.219

### WHICH SELLERS HAVE THE SLOWEST DELIVERY RATE?


In [8]:
WITH time_lag AS (
    SELECT
    order_id,
    DATEDIFF(DAY,order_purchase_timestamp, order_delivered_customer_date) AS time_delays
FROM olist_orders_dataset
WHERE order_status = 'delivered'
AND order_delivered_customer_date IS NOT NULL
)
SELECT TOP 10
oi.seller_id,
SUM(tl.time_delays) AS total_delays,
DENSE_RANK() OVER (ORDER BY SUM(tl.time_delays) DESC) AS rank
FROM time_lag AS tl
JOIN olist_order_items_dataset oi
ON tl.order_id=oi.order_id
GROUP BY oi.seller_id
ORDER BY total_delays DESC;

(10 rows affected)

seller_id                        | total_delays | rank
---------------------------------+--------------+-----
7c67e1448b00f6e969d365cea6b010ab | 30262        | 1   
1f50f920176fa81dab994f9023523100 | 29816        | 2   
4a3ca9315b744ce9f8e9374361493884 | 28007        | 3   
cc419e0650a3c5ba77189a1882b7556a | 19743        | 4   
6560211a19b47992c3666cc44a7e94c0 | 18906        | 5   
da8622b14eb17ae2831f4ac5b9dab84a | 17202        | 6   
4869f7a5dfa277a7dca6462dcf3b52b2 | 17147        | 7   
1025f0e2d44d7041d6cf58b6550e0bfa | 17036        | 8   
ea8482cd71df3c1969d7b9473ff13abc | 15705        | 9   
955fee9216a65b617aa5c0531780ce60 | 15703        | 10  
(10 rows)

Total execution time: 00:00:00.339

### HOW DID OLIST'S MONTHLY SALES CHANGED COMPARED TO PREVIOUS MONTH?

In [19]:
WITH sales_by_month AS 
((
    SELECT
    DATEFROMPARTS(YEAR(op.order_purchase_timestamp),
    MONTH(op.order_purchase_timestamp),1) AS sales_month,
    ROUND(SUM(oi.price),2) AS total_sales
    FROM olist_orders_dataset op
    JOIN olist_order_items_dataset oi
    ON op.order_id=oi.order_id
    GROUP BY  DATEFROMPARTS(YEAR(op.order_purchase_timestamp),MONTH(op.order_purchase_timestamp),1)))
SELECT
sales_month,
total_sales,
LAG(total_sales) OVER (ORDER BY sales_month) AS sales_change
FROM sales_by_month
ORDER BY sales_month;
        

(24 rows affected)

sales_month | total_sales | sales_change
------------+-------------+-------------
2016-09-01  | 267.36      | NULL        
2016-10-01  | 49507.66    | 267.36      
2016-12-01  | 10.9        | 49507.66    
2017-01-01  | 120312.87   | 10.9        
2017-02-01  | 247303.02   | 120312.87   
2017-03-01  | 374344.3    | 247303.02   
2017-04-01  | 359927.23   | 374344.3    
2017-05-01  | 506071.14   | 359927.23   
2017-06-01  | 433038.6    | 506071.14   
2017-07-01  | 498031.48   | 433038.6    
2017-08-01  | 573971.68   | 498031.48   
2017-09-01  | 624401.69   | 573971.68   
2017-10-01  | 664219.43   | 624401.69   
2017-11-01  | 1010271.37  | 664219.43   
2017-12-01  | 743914.17   | 1010271.37  
2018-01-01  | 950030.36   | 743914.17   
2018-02-01  | 844178.71   | 950030.36   
2018-03-01  | 983213.44   | 844178.71   
2018-04-01  | 996647.75   | 983213.44   
2018-05-01  | 996517.68   | 996647.75   
2018-06-01  | 865124.31   | 996517.68   
2018-07-01  | 895507.22   | 865124.31

### WHICH CUSTOMERS HAVE REPEAT PURCHASES?

In [21]:
SELECT
customer_unique_id,
COUNT(DISTINCT op.order_id) AS total_orders
FROM olist_customers_dataset oc
JOIN olist_orders_dataset op
ON oc.customer_id=op.customer_id
GROUP BY customer_unique_id
HAVING COUNT(DISTINCT op.order_id) > 1
ORDER BY total_orders DESC;

(2997 rows affected)

customer_unique_id               | total_orders
---------------------------------+-------------
8d50f5eadf50201ccdcedfb9e2ac8455 | 17          
3e43e6105506432c953e165fb2acf44c | 9           
6469f99c1f9dfae7733b25662e7f1782 | 7           
1b6c7548a2a1f9037c1fd3ddfed95f33 | 7           
ca77025e7201e3b30c44b472ff346268 | 7           
12f5d6e1cbf93dafd9dcc19095df0b3d | 6           
dc813062e0fc23409cd255f7f53c7074 | 6           
de34b16117594161a6a89c50b289d35a | 6           
47c1a3033b8b77b3ab6e109eb4d5fdf3 | 6           
f0e310a6839dce9de1638e0fe5ab282a | 6           
63cfc61cee11cbe306bff5857d00bfe4 | 6           
b4e4f24de1e8725b74e4a1f4975116ed | 5           
56c8638e7c058b98aae6d74d2dd6ea23 | 5           
4e65032f1f574189fb793bac5a867bbc | 5           
394ac4de8f3acb14253c177f0e15bc58 | 5           
fe81bb32c243a86b2f86fbf053fe6140 | 5           
5e8f38a9a1c023f3db718edcf926a2db | 5           
35ecdf6858edc6427223b64804cf028e | 5           
74cb1ad7e6d5674325

### WHICH MONTHS HAVE HIGHEST PERCENTAGE OF CANCELLATION RATE?

In [24]:
SELECT 
YEAR(order_purchase_timestamp) AS order_year,
MONTH(order_purchase_timestamp) AS order_month,
COUNT(order_id) AS total_orders,
SUM(CASE WHEN order_status = 'canceled' THEN 1 ELSE 0 END) AS order_canceled,
100* SUM(CASE WHEN order_status = 'canceled' THEN 1 ELSE 0 END) /COUNT(order_id) AS cancellation_rate
FROM olist_orders_dataset
GROUP BY YEAR(order_purchase_timestamp), MONTH(order_purchase_timestamp)
ORDER BY cancellation_rate DESC;

(25 rows affected)

order_year | order_month | total_orders | order_canceled | cancellation_rate
-----------+-------------+--------------+----------------+------------------
2018       | 10          | 4            | 4              | 100              
2018       | 9           | 16           | 15             | 93               
2016       | 9           | 4            | 2              | 50               
2016       | 10          | 324          | 24             | 7                
2018       | 2           | 6728         | 73             | 1                
2017       | 3           | 2682         | 33             | 1                
2018       | 8           | 6512         | 84             | 1                
2017       | 2           | 1780         | 17             | 0                
2018       | 6           | 6167         | 18             | 0                
2017       | 9           | 4285         | 20             | 0                
2017       | 6           | 3245         | 16            